⚙️ Configuração do ambiente

Este notebook foi desenvolvido para execução em Google Colab com acelerador de hardware GPU.

Antes de executar as células:

Acesse Ambiente de execução → Alterar tipo de ambiente de execução.
Em Acelerador de hardware, selecione GPU.
Salve a configuração e execute o notebook normalmente.

Importante: o treinamento do modelo de linguagem requer GPU. A execução em CPU não é recomendada e pode inviabilizar ou tornar o treinamento excessivamente lento.

In [ ]:
#Instalações
!pip install -q -U \
    transformers \
    datasets \
    peft \
    trl \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    langchain \
    langchain-core \
    langchain-groq \
    langgraph \
    groq \
    python-dotenv
!pip install -q evaluate rouge_score
!pip install -q langchain-huggingface langchain-chroma
!pip install -q \
    "opentelemetry-api==1.27.0" \
    "opentelemetry-sdk==1.27.0" \
    "opentelemetry-exporter-otlp-proto-grpc==1.27.0"
!pip uninstall -y wandb


In [ ]:
# ==========================================
# BIBLIOTECAS GERAIS
# ==========================================

import os
os.environ["WANDB_DISABLED"] = "true"

import json
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch


from collections import Counter

# ==========================================
# ACESSO E PROCESSAMENTO DO MEDQUAD
# ==========================================

import requests
import xml.etree.ElementTree as ET


# ==========================================
# HUGGING FACE / FINE-TUNING
# ==========================================

import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer


# ==========================================
# LANGCHAIN
# ==========================================

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# ==========================================
# LANGGRAPH
# ==========================================

from langgraph.graph import StateGraph, END


# ==========================================
# GROQ
# ==========================================

from groq import Groq


# ==========================================
# CONFIGURAÇÕES
# ==========================================

from dotenv import load_dotenv

##Fine-tunning - Pubmedqa
Tem a opção de fazer o download por arquivo local ou conectar diretamente no Github

In [ ]:
# ============================================================
# CARREGAMENTO DO PUBMEDQA
# ============================================================

# Opção 1 — arquivo local (recomendado para reprodutibilidade)
with open("ori_pqal.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Registros carregados: {len(data)}")


# Opção 2 — carregamento direto do GitHub
# Descomente este bloco caso queira atualizar diretamente do repositório.

# url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json"

# response = requests.get(url)
# response.raise_for_status()

# data = response.json()

# print(f"Registros carregados: {len(data)}")

Registros carregados: 1000


In [ ]:
# ============================================================
# ANÁLISE INICIAL DO DATASET - PUBMEDQA
# ============================================================

registros = list(data.values())

print("=== VISÃO GERAL DO DATASET ===")
print(f"Total de registros: {len(registros)}")

# ------------------------------------------------------------
# 1. COMPLETUDE DOS DADOS
# ------------------------------------------------------------

campos_principais = ["QUESTION", "CONTEXTS", "LONG_ANSWER"]

print("\n=== COMPLETUDE ===")

for campo in campos_principais:
    vazios = sum(
        1 for registro in registros
        if not registro[campo]
    )
    print(f"{campo}: {vazios} vazios")

anos_ausentes = sum(
    1 for registro in registros
    if not registro["YEAR"]
)

print(f"YEAR: {anos_ausentes} ausentes")


# ------------------------------------------------------------
# 2. DUPLICIDADES
# ------------------------------------------------------------

perguntas = [registro["QUESTION"] for registro in registros]
respostas = [registro["LONG_ANSWER"] for registro in registros]

print("\n=== DUPLICIDADES ===")
print(f"Perguntas duplicadas: {len(perguntas) - len(set(perguntas))}")
print(f"Respostas duplicadas: {len(respostas) - len(set(respostas))}")


# ------------------------------------------------------------
# 3. DISTRIBUIÇÃO DAS DECISÕES
# ------------------------------------------------------------

decisoes = Counter(
    registro["final_decision"]
    for registro in registros
)

print("\n=== DISTRIBUIÇÃO DE FINAL_DECISION ===")

for decisao, quantidade in sorted(decisoes.items()):
    percentual = quantidade / len(registros) * 100
    print(f"{decisao}: {quantidade} ({percentual:.2f}%)")


# ------------------------------------------------------------
# 4. TAMANHO DOS TEXTOS
# ------------------------------------------------------------

perguntas_palavras = [
    len(registro["QUESTION"].split())
    for registro in registros
]

respostas_palavras = [
    len(registro["LONG_ANSWER"].split())
    for registro in registros
]

contextos_palavras = [
    len(" ".join(registro["CONTEXTS"]).split())
    for registro in registros
]

quantidade_contextos = [
    len(registro["CONTEXTS"])
    for registro in registros
]

print("\n=== TAMANHO DOS TEXTOS (PALAVRAS) ===")

print(
    f"QUESTION: média {np.mean(perguntas_palavras):.1f} | "
    f"máximo {max(perguntas_palavras)}"
)

print(
    f"CONTEXTS: média {np.mean(contextos_palavras):.1f} | "
    f"máximo {max(contextos_palavras)}"
)

print(
    f"LONG_ANSWER: média {np.mean(respostas_palavras):.1f} | "
    f"máximo {max(respostas_palavras)}"
)

print(
    f"Contextos por registro: média {np.mean(quantidade_contextos):.2f} | "
    f"máximo {max(quantidade_contextos)}"
)


# ------------------------------------------------------------
# 5. CONSISTÊNCIA ENTRE CONTEXTS E LABELS
# ------------------------------------------------------------

inconsistentes = sum(
    len(registro["CONTEXTS"]) != len(registro["LABELS"])
    for registro in registros
)

print("\n=== CONSISTÊNCIA ===")
print(f"CONTEXTS x LABELS inconsistentes: {inconsistentes}")

=== VISÃO GERAL DO DATASET ===
Total de registros: 1000

=== COMPLETUDE ===
QUESTION: 0 vazios
CONTEXTS: 0 vazios
LONG_ANSWER: 0 vazios
YEAR: 58 ausentes

=== DUPLICIDADES ===
Perguntas duplicadas: 0
Respostas duplicadas: 0

=== DISTRIBUIÇÃO DE FINAL_DECISION ===
maybe: 110 (11.00%)
no: 338 (33.80%)
yes: 552 (55.20%)

=== TAMANHO DOS TEXTOS (PALAVRAS) ===
QUESTION: média 12.9 | máximo 31
CONTEXTS: média 200.2 | máximo 398
LONG_ANSWER: média 39.7 | máximo 126
Contextos por registro: média 3.36 | máximo 9

=== CONSISTÊNCIA ===
CONTEXTS x LABELS inconsistentes: 0


In [ ]:
# ============================================================
# CRIAÇÃO DO DATAFRAME
# ============================================================

import pandas as pd

df_pubmed = pd.DataFrame([
    {
        "question": registro["QUESTION"],
        "contexts": registro["CONTEXTS"],
        "answer": registro["LONG_ANSWER"],
        "decision": registro["final_decision"]
    }
    for registro in registros
])

df_pubmed.head()

,question,contexts,answer,decision
0,Do mitochondria play a role in remodelling lac...,[Programmed cell death (PCD) is the regulated ...,Results depicted mitochondrial dynamics in viv...,yes
1,Landolt C and snellen e acuity: differences in...,[Assessment of visual acuity depends on the op...,"Using the charts described, there was only a s...",no
2,"Syncope during bathing in infants, a pediatric...",[Apparent life-threatening events in infants a...,"""Aquagenic maladies"" could be a pediatric form...",yes
3,Are the long-term results of the transanal pul...,[The transanal endorectal pull-through (TERPT)...,Our long-term study showed significantly bette...,no
4,Can tailored interventions increase mammograph...,[Telephone counseling and tailored print commu...,The effects of the intervention were most pron...,yes


In [ ]:
# ============================================================
# ETAPA 3 — PREPROCESSAMENTO E CURADORIA
# 3.1 — Unificação e limpeza dos textos
# ============================================================

def limpar_texto(texto):
    return " ".join(str(texto).split())


df_pubmed["question"] = df_pubmed["question"].apply(limpar_texto)
df_pubmed["answer"] = df_pubmed["answer"].apply(limpar_texto)

df_pubmed["context"] = df_pubmed["contexts"].apply(
    lambda contextos: " ".join(limpar_texto(contexto) for contexto in contextos)
)

df_pubmed.head()

,question,contexts,answer,decision,context
0,Do mitochondria play a role in remodelling lac...,[Programmed cell death (PCD) is the regulated ...,Results depicted mitochondrial dynamics in viv...,yes,Programmed cell death (PCD) is the regulated d...
1,Landolt C and snellen e acuity: differences in...,[Assessment of visual acuity depends on the op...,"Using the charts described, there was only a s...",no,Assessment of visual acuity depends on the opt...
2,"Syncope during bathing in infants, a pediatric...",[Apparent life-threatening events in infants a...,"""Aquagenic maladies"" could be a pediatric form...",yes,Apparent life-threatening events in infants ar...
3,Are the long-term results of the transanal pul...,[The transanal endorectal pull-through (TERPT)...,Our long-term study showed significantly bette...,no,The transanal endorectal pull-through (TERPT) ...
4,Can tailored interventions increase mammograph...,[Telephone counseling and tailored print commu...,The effects of the intervention were most pron...,yes,Telephone counseling and tailored print commun...


In [ ]:
# ============================================================
# ETAPA 3.2 — VERIFICAÇÃO DA QUALIDADE DOS DADOS
# ============================================================

print("=== VERIFICAÇÃO DA QUALIDADE ===")

print(f"Total de registros: {len(df_pubmed)}")
print(f"Perguntas vazias: {df_pubmed['question'].eq('').sum()}")
print(f"Contextos vazios: {df_pubmed['context'].eq('').sum()}")
print(f"Respostas vazias: {df_pubmed['answer'].eq('').sum()}")

print("\n=== TAMANHO DOS TEXTOS ===")

print(
    f"Pergunta - média: "
    f"{df_pubmed['question'].str.split().str.len().mean():.1f} palavras"
)

print(
    f"Contexto - média: "
    f"{df_pubmed['context'].str.split().str.len().mean():.1f} palavras"
)

print(
    f"Resposta - média: "
    f"{df_pubmed['answer'].str.split().str.len().mean():.1f} palavras"
)

=== VERIFICAÇÃO DA QUALIDADE ===
Total de registros: 1000
Perguntas vazias: 0
Contextos vazios: 0
Respostas vazias: 0

=== TAMANHO DOS TEXTOS ===
Pergunta - média: 12.9 palavras
Contexto - média: 200.2 palavras
Resposta - média: 39.7 palavras


In [ ]:
# ============================================================
# ETAPA 3.3 — VERIFICAÇÃO DE POSSÍVEIS DADOS PESSOAIS (PII)
# ============================================================

import re

texto_pubmed = (
    df_pubmed["question"] + " " +
    df_pubmed["context"] + " " +
    df_pubmed["answer"]
)

padroes_pii = {
    "emails": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
    "telefones": r"\b(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,3}\)?[\s.-]?)?\d{4,5}[\s.-]?\d{4}\b",
    "urls": r"https?://\S+|www\.\S+"
}

print("=== VERIFICAÇÃO DE POSSÍVEIS PII ===")

for nome, padrao in padroes_pii.items():
    ocorrencias = texto_pubmed.str.findall(padrao)
    total = ocorrencias.str.len().sum()

    print(f"{nome}: {total} ocorrência(s)")
#Foi realizada uma verificação automatizada para identificar possíveis dados pessoais,
#como e-mails, telefones e URLs. As ocorrências identificadas foram analisadas individualmente
#e correspondiam a intervalos de anos, períodos de estudos, referências de páginas científicas
#ou referências bibliográficas. Dessa forma, não foram identificados dados pessoais que
#demandassem tratamento ou anonimização no conjunto de dados utilizado,
#não sendo necessária a aplicação de técnicas de anonimização.

=== VERIFICAÇÃO DE POSSÍVEIS PII ===
emails: 0 ocorrência(s)
telefones: 44 ocorrência(s)
urls: 1 ocorrência(s)


In [ ]:
# ============================================================
# ETAPA 4 — PREPARAÇÃO DOS DADOS PARA O FINE-TUNING
# 4.1 — Divisão dos dados em treino, validação e teste
# ============================================================

from sklearn.model_selection import train_test_split

# Primeiro: 70% treino e 30% temporário
df_treino, df_temporario = train_test_split(
    df_pubmed,
    test_size=0.30,
    stratify=df_pubmed["decision"],
    random_state=42
)

# Segundo: divisão dos 30% restantes em
# 20% validação e 10% teste
df_validacao, df_teste = train_test_split(
    df_temporario,
    test_size=1/3,
    stratify=df_temporario["decision"],
    random_state=42
)

print("=== DIVISÃO DOS DADOS ===")
print(f"Total de registros: {len(df_pubmed)}")
print(f"Treino: {len(df_treino)}")
print(f"Validação: {len(df_validacao)}")
print(f"Teste: {len(df_teste)}")

=== DIVISÃO DOS DADOS ===
Total de registros: 1000
Treino: 700
Validação: 200
Teste: 100


In [ ]:
# ============================================================
# Formatação dos exemplos para SFT
# ============================================================

def criar_exemplo_sft(linha):
    return (
        f"Contexto: {linha['context']}\n\n"
        f"Pergunta: {linha['question']}\n\n"
        f"Resposta: {linha['answer']}"
    )

df_treino["texto_sft"] = df_treino.apply(criar_exemplo_sft, axis=1)
df_validacao["texto_sft"] = df_validacao.apply(criar_exemplo_sft, axis=1)
df_teste["texto_sft"] = df_teste.apply(criar_exemplo_sft, axis=1)

df_treino[["question", "answer", "texto_sft"]].head(2)

,question,answer,texto_sft
310,Telemedicine and type 1 diabetes: is technolog...,The Diabeo system improved glycaemic control i...,Contexto: Each patient received a smartphone w...
830,Could chest wall rigidity be a factor in rapid...,In summary we believe sudden onset chest wall ...,Contexto: There has been a significant spike i...


In [ ]:
# ============================================================
# ETAPA 4.1 — VERIFICAÇÃO DA DISTRIBUIÇÃO DOS CONJUNTOS
# ============================================================

print("=== DISTRIBUIÇÃO POR DECISÃO ===")

print("\nTreino:")
print(df_treino["decision"].value_counts(normalize=True).round(3))

print("\nValidação:")
print(df_validacao["decision"].value_counts(normalize=True).round(3))

print("\nTeste:")
print(df_teste["decision"].value_counts(normalize=True).round(3))

=== DISTRIBUIÇÃO POR DECISÃO ===

Treino:
decision
yes      0.551
no       0.339
maybe    0.110
Name: proportion, dtype: float64

Validação:
decision
yes      0.555
no       0.335
maybe    0.110
Name: proportion, dtype: float64

Teste:
decision
yes      0.55
no       0.34
maybe    0.11
Name: proportion, dtype: float64


###Modelo LORA

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer carregado com sucesso!")
print("Tamanho do vocabulário:", tokenizer.vocab_size)

Tokenizer carregado com sucesso!
Tamanho do vocabulário: 151643


In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

modelo = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Modelo carregado com sucesso!")
print(modelo.device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modelo carregado com sucesso!
cpu


In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

print("Configuração LoRA criada com sucesso!")

Configuração LoRA criada com sucesso!


In [ ]:
from peft import get_peft_model

modelo = get_peft_model(modelo, lora_config)

modelo.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [ ]:
#================================
#CRIAR DATA SET
#================================

from datasets import Dataset

dataset_treino = Dataset.from_pandas(
    df_treino[["texto_sft"]],
    preserve_index=False
)

dataset_validacao = Dataset.from_pandas(
    df_validacao[["texto_sft"]],
    preserve_index=False
)

dataset_teste = Dataset.from_pandas(
    df_teste[["texto_sft"]],
    preserve_index=False
)

print(dataset_treino)
print(dataset_validacao)
print(dataset_teste)

Dataset({
    features: ['texto_sft'],
    num_rows: 700
})
Dataset({
    features: ['texto_sft'],
    num_rows: 200
})
Dataset({
    features: ['texto_sft'],
    num_rows: 100
})


In [ ]:
def tokenizar_dados(exemplos):
    return tokenizer(
        exemplos["texto_sft"],
        truncation=True,
        max_length=512,
        padding=False
    )


dataset_treino_tokenizado = dataset_treino.map(
    tokenizar_dados,
    batched=True
)

dataset_validacao_tokenizado = dataset_validacao.map(
    tokenizar_dados,
    batched=True
)

dataset_teste_tokenizado = dataset_teste.map(
    tokenizar_dados,
    batched=True
)

print(dataset_treino_tokenizado)

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['texto_sft', 'input_ids', 'attention_mask'],
    num_rows: 700
})


In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator configurado com sucesso.")

Data collator configurado com sucesso.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./resultado_finetuning",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none"
)

print("TrainingArguments configurado com sucesso.")

TrainingArguments configurado com sucesso.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=modelo,
    args=training_args,
    train_dataset=dataset_treino_tokenizado,
    eval_dataset=dataset_validacao_tokenizado,
    data_collator=data_collator
)

print("Trainer configurado com sucesso.")

Trainer configurado com sucesso.


In [21]:
resultado_treinamento = trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
resultado_teste = trainer.evaluate(
    dataset_teste_tokenizado
)

print(resultado_teste)

In [ ]:
import torch

modelo.eval()

def gerar_resposta(pergunta, contexto):
    texto = f"""Contexto: {contexto}

Pergunta: {pergunta}

Resposta:"""

    entradas = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(modelo.device)

    with torch.no_grad():
        saida = modelo.generate(
            **entradas,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    resposta = tokenizer.decode(
        saida[0][entradas["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return resposta.strip()

In [ ]:
exemplo_teste = df_teste.iloc[0]

resposta_gerada = gerar_resposta(
    exemplo_teste["question"],
    exemplo_teste["context"]
)

print("Pergunta:")
print(exemplo_teste["question"])

print("\nResposta esperada:")
print(exemplo_teste["answer"])

print("\nResposta gerada pelo modelo:")
print(resposta_gerada)

In [ ]:
resultados_teste = []

for _, exemplo in df_teste.iterrows():
    resposta = gerar_resposta(
        exemplo["question"],
        exemplo["context"]
    )

    resultados_teste.append({
        "question": exemplo["question"],
        "decision_esperada": exemplo["decision"],
        "answer_esperada": exemplo["answer"],
        "answer_gerada": resposta
    })

df_resultados_teste = pd.DataFrame(resultados_teste)

print(f"Exemplos avaliados: {len(df_resultados_teste)}")

In [ ]:
for i in range(5):
    exemplo = df_resultados_teste.iloc[i]

    print(f"\n{'='*80}")
    print(f"EXEMPLO {i+1}")
    print(f"{'='*80}")

    print("\nPERGUNTA:")
    print(exemplo["question"])

    print("\nDECISÃO ESPERADA:")
    print(exemplo["decision_esperada"])

    print("\nRESPOSTA ESPERADA:")
    print(exemplo["answer_esperada"])

    print("\nRESPOSTA GERADA:")
    print(exemplo["answer_gerada"])

O modelo apresentou capacidade de gerar respostas coerentes e relacionadas ao contexto médico após o fine-tuning. Entretanto, foram observadas limitações de fidelidade ao conteúdo de referência, geração de informações adicionais e presença de artefatos textuais provenientes dos dados de treinamento. Dessa forma, os resultados demonstram a capacidade do modelo de adaptação ao domínio, mas não permitem considerá-lo adequado para utilização clínica sem validação especializada.

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

resultados_rouge = rouge.compute(
    predictions=df_resultados_teste["answer_gerada"].tolist(),
    references=df_resultados_teste["answer_esperada"].tolist()
)

print("=== RESULTADOS ROUGE ===")
for metrica, valor in resultados_rouge.items():
    print(f"{metrica}: {valor:.4f}")

In [ ]:
def classificar_decisao(pergunta, contexto):
    texto = f"""Contexto: {contexto}

Pergunta: {pergunta}

Classifique a resposta em apenas uma das opções:
yes
no
maybe

Resposta:"""

    entradas = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(modelo.device)

    with torch.no_grad():
        saida = modelo.generate(
            **entradas,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    resposta = tokenizer.decode(
        saida[0][entradas["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip().lower()

    for decisao in ["yes", "no", "maybe"]:
        if decisao in resposta:
            return decisao

    return "indefinido"


print("Função de classificação criada.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

decisoes_geradas = []

for _, exemplo in df_teste.iterrows():
    decisao = classificar_decisao(
        exemplo["question"],
        exemplo["context"]
    )

    decisoes_geradas.append(decisao)

df_resultados_teste["decision_gerada"] = decisoes_geradas

acuracia = accuracy_score(
    df_resultados_teste["decision_esperada"],
    df_resultados_teste["decision_gerada"]
)

print(f"Acurácia: {acuracia:.4f}")

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(
    classification_report(
        df_resultados_teste["decision_esperada"],
        df_resultados_teste["decision_gerada"],
        labels=["yes", "no", "maybe"],
        zero_division=0
    )
)

In [ ]:
print("=== RESUMO DO FINE-TUNING ===")

print(f"Parâmetros treináveis: 4.358.144")
print(f"Total de parâmetros: 1.548.072.448")
print(f"Percentual treinável: 0,2815%")

print("\nÉpocas executadas: 3")
print("Batch size: 1")
print("Gradient accumulation: 4")
print("Learning rate: 2e-4")
print("Melhor época: 1")
print("Melhor validation loss: 1,8092")
print("Test loss: 1,8305")

In [ ]:
print("Melhor checkpoint:")
print(trainer.state.best_model_checkpoint)

print("\nMelhor evaluation loss:")
print(trainer.state.best_metric)

In [ ]:
#===================
# SALVAR LORA
#====================

modelo.save_pretrained("./modelo_lora_pubmedqa")
tokenizer.save_pretrained("./modelo_lora_pubmedqa")

print("Modelo LoRA e tokenizer salvos com sucesso.")

In [ ]:
pergunta_nova = """
Does regular physical activity improve quality of life in patients with chronic disease?
"""

contexto_novo = """
Regular physical activity has been associated with improvements in physical
function, mental health, and quality of life in patients with several chronic
diseases. The magnitude of benefit may vary according to the disease,
intensity of activity, and individual patient characteristics.
"""

resposta_nova = gerar_resposta(
    pergunta_nova,
    contexto_novo
)

print("PERGUNTA:")
print(pergunta_nova.strip())

print("\nCONTEXTO:")
print(contexto_novo.strip())

print("\nRESPOSTA GERADA PELO MODELO:")
print(resposta_nova)

In [ ]:
contexto_controlado = """
Studies suggest that regular physical activity can improve quality of life
in patients with chronic diseases.
"""

resposta_controlada = gerar_resposta(
    pergunta_nova,
    contexto_controlado
)

print("RESPOSTA GERADA:")
print(resposta_controlada)

O modelo apresentou boa capacidade de geração de texto no domínio médico, porém demonstrou tendência à alucinação, especialmente quando recebeu contextos curtos. Assim, as respostas não devem ser consideradas clinicamente confiáveis sem validação por fontes e profissionais especializados.
Limitação da LLM: apesar da especialização obtida por fine-tuning, o modelo permanece sujeito às limitações inerentes a modelos generativos, como geração de informações não explicitamente presentes no contexto e reprodução de padrões estatisticamente plausíveis. No domínio médico, esse comportamento exige validação das respostas e não permite considerar o modelo como fonte autônoma de informação clínica.

## Importar aquivo para RAG - Medquad

In [ ]:
# URL da pasta do MedQuAD no Git
MEDQUAD_URL = (
    "https://api.github.com/repos/"
    "abachaa/MedQuAD/contents/4_MPlus_Health_Topics_QA"
)

response = requests.get(MEDQUAD_URL)

response.raise_for_status()

arquivos_medquad = response.json()

print(f"Arquivos encontrados: {len(arquivos_medquad)}")

# Mostrar os primeiros arquivos encontrados

for arquivo in arquivos_medquad[:10]:
    print(arquivo["name"])

#Verificar se todos os arquivos são xml

arquivos_xml = [
    arquivo
    for arquivo in arquivos_medquad
    if arquivo["name"].lower().endswith(".xml")
]

print(f"Arquivos XML encontrados: {len(arquivos_xml)}")

In [ ]:
dados_medquad = []

for arquivo in arquivos_xml:

    response = requests.get(arquivo["download_url"])
    response.raise_for_status()

    root = ET.fromstring(response.content)

    focus = root.findtext("Focus")

    for qa_pair in root.findall(".//QAPair"):

        question_element = qa_pair.find("Question")
        answer_element = qa_pair.find("Answer")

        dados_medquad.append({
            "qid": question_element.attrib.get("qid"),
            "qtype": question_element.attrib.get("qtype"),
            "focus": focus,
            "question": question_element.text,
            "answer": answer_element.text
        })

df_medquad = pd.DataFrame(dados_medquad)

print("Formato:", df_medquad.shape)
print("\nPrimeiros registros:")
display(df_medquad.head())

In [ ]:
#Entendimento do DF
df_medquad.info()

print("=== DUPLICIDADES ===")
print("Linhas duplicadas:", df_medquad.duplicated().sum())
print("Perguntas duplicadas:", df_medquad["question"].duplicated().sum())
print("Respostas duplicadas:", df_medquad["answer"].duplicated().sum())

print("\n=== TIPOS DE PERGUNTA ===")
print(df_medquad["qtype"].value_counts())

print("\n=== TAMANHO DOS TEXTOS ===")
print("Pergunta - média:", round(df_medquad["question"].str.len().mean(), 2))
print("Pergunta - mínimo:", df_medquad["question"].str.len().min())
print("Pergunta - máximo:", df_medquad["question"].str.len().max())

print("\nResposta - média:", round(df_medquad["answer"].str.len().mean(), 2))
print("Resposta - mínimo:", df_medquad["answer"].str.len().min())
print("Resposta - máximo:", df_medquad["answer"].str.len().max())

In [ ]:
df_medquad['question_pattern'] = df_medquad.apply(
    lambda row: row['question'].replace(row['focus'], '[FOCUS]'),
    axis=1
)

df_medquad['question_pattern'].value_counts()

In [ ]:
#===========================
# TRANSFORMAR MEDQUAD EM BASE DE CONHECIMENTO - RAG
#==========================================================


from langchain_core.documents import Document

documentos_medquad = []

for _, registro in df_medquad.iterrows():
    documentos_medquad.append(
        Document(
            page_content=(
                f"Pergunta: {registro['question']}\n"
                f"Resposta: {registro['answer']}"
            ),
            metadata={
                "qid": registro["qid"],
                "qtype": registro["qtype"],
                "focus": registro["focus"],
                "question_pattern": registro["question_pattern"]
            }
        )
    )

print(f"Documentos criados: {len(documentos_medquad)}")
print("\nExemplo:")
print(documentos_medquad[0])

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Modelo de embeddings carregado com sucesso.")

In [ ]:
from langchain_chroma import Chroma

banco_medquad = Chroma.from_documents(
    documents=documentos_medquad,
    embedding=embeddings,
    collection_name="medquad"
)

print("Banco vetorial criado com sucesso.")
print(f"Documentos indexados: {len(documentos_medquad)}")

In [ ]:
resultado_busca = banco_medquad.similarity_search(
    "What are the symptoms of breast cancer?",
    k=1
)

for i, documento in enumerate(resultado_busca, 1):
    print(f"\n--- Resultado {i} ---")
    print(documento.page_content[:1000])

###Vincular Lora, Tokenizer e Langchain

In [ ]:
print(type(modelo))
print(type(tokenizer))

In [ ]:
from transformers import pipeline, GenerationConfig
from langchain_huggingface import HuggingFacePipeline

generation_config = GenerationConfig.from_model_config(modelo.config)

generation_config.max_length = None
generation_config.max_new_tokens = 128
generation_config.temperature = 0.1
generation_config.top_p = 0.9
generation_config.do_sample = True
generation_config.pad_token_id = tokenizer.eos_token_id

pipeline_geracao = pipeline(
    "text-generation",
    model=modelo,
    tokenizer=tokenizer,
    generation_config=generation_config,
    return_full_text=False
)

llm_medica = HuggingFacePipeline(
    pipeline=pipeline_geracao
)

print("Pipeline de geração configurada com sucesso.")
print(pipeline_geracao.generation_config)

In [ ]:
retriever_medquad = banco_medquad.as_retriever(
    search_kwargs={"k": 1}
)

print("Retriever criado com sucesso.")

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_medico = PromptTemplate(
    input_variables=["contexto", "pergunta"],
    template="""Você é um assistente de informação médica.

Sua tarefa é responder à pergunta do usuário utilizando duas fontes:
1. o conhecimento adquirido durante o fine-tuning;
2. o contexto médico recuperado pela busca RAG.

FLUXO DE RESPOSTA:

1. INTERPRETAÇÃO
Identifique exatamente o que está sendo solicitado na pergunta.

2. CONHECIMENTO DO FINE-TUNING
Utilize somente informações relacionadas ao tema que sejam compatíveis
com o conhecimento adquirido durante o fine-tuning.

3. BASE DE CONHECIMENTO RAG
Considere somente informações presentes explicitamente no contexto
médico recuperado.

4. VALIDAÇÃO
Antes de responder, verifique se as informações utilizadas na resposta
podem ser sustentadas pelo conhecimento do fine-tuning ou pelo contexto RAG.

5. RESTRIÇÕES
- Não utilize conhecimento médico externo às duas fontes.
- Não invente informações.
- Não faça inferências médicas que não sejam sustentadas pelas fontes.
- Não altere o significado das informações das fontes.
- Não acrescente informações que não sejam necessárias para responder
  à pergunta.
- Se houver informações no contexto sobre outras condições, ignore-as.

6. RESPOSTA
Responda somente à pergunta do usuário, de forma objetiva e clara.
Responda sempre em português do Brasil.

7. INSUFICIÊNCIA
Se as fontes disponíveis não fornecerem informação suficiente para
responder à pergunta, responda exatamente:

"Não há informações suficientes nas fontes disponíveis para responder
a esta pergunta."

TEXTO DE REFERÊNCIA DO RAG:
{contexto}

PERGUNTA DO USUÁRIO:
{pergunta}

RESPOSTA:"""
)

In [ ]:
def assistente_medico(pergunta):
    pergunta_busca = traduzir_pergunta(pergunta)

    documentos = buscar_medquad(
        pergunta_busca,
        k=5
    )

    documento_principal = documentos[0]

    contexto = documento_principal.page_content

    resposta = llm_medica.invoke(
        prompt_medico.format(
            contexto=contexto,
            pergunta=pergunta
        )
    )

    fontes = []

    qid = documento_principal.metadata.get("qid", "N/A")
    focus = documento_principal.metadata.get("focus", "N/A")

    fontes.append(
        f"MedQuAD — {focus} (qid: {qid})"
    )

    return resposta, fontes

def traduzir_pergunta(pergunta):
    resposta = llm_medica.invoke(
        prompt_traducao.format(pergunta=pergunta)
    )

    return resposta.strip()


print("Função de tradução criada com sucesso.")

from langchain_core.prompts import PromptTemplate

prompt_traducao = PromptTemplate(
    input_variables=["pergunta"],
    template="""Traduza a pergunta médica abaixo do português para o inglês.

Regras:
- Retorne SOMENTE a tradução.
- Não explique a tradução.
- Preserve o significado médico da pergunta.

Pergunta:
{pergunta}

Tradução:"""
)

print("Prompt de tradução criado com sucesso.")

In [ ]:
pergunta = "Quais são os sintomas do câncer de mama?"

pergunta_busca = traduzir_pergunta(pergunta)

documentos = buscar_medquad(
    pergunta_busca,
    k=5
)

documento_principal = documentos[0]

contexto = documento_principal.page_content

prompt_final = prompt_medico.format(
    contexto=contexto,
    pergunta=pergunta
)

print(prompt_final)

In [ ]:
# Corrige a configuração de geração herdada pela pipeline

pipeline_geracao.generation_config.max_length = None
pipeline_geracao.generation_config.max_new_tokens = 128
pipeline_geracao.generation_config.temperature = 0.1
pipeline_geracao.generation_config.top_p = 0.9
pipeline_geracao.generation_config.do_sample = True
pipeline_geracao.generation_config.pad_token_id = tokenizer.eos_token_id

print(pipeline_geracao.generation_config)

In [ ]:
from transformers import GenerationConfig, pipeline
from langchain_huggingface import HuggingFacePipeline

config_traducao = GenerationConfig.from_model_config(modelo.config)

config_traducao.max_length = None
config_traducao.max_new_tokens = 64
config_traducao.do_sample = False
config_traducao.pad_token_id = tokenizer.eos_token_id

# Remove parâmetros de amostragem que não são usados na tradução
config_traducao.temperature = None
config_traducao.top_p = None
config_traducao.top_k = None

pipeline_traducao = pipeline(
    "text-generation",
    model=modelo,
    tokenizer=tokenizer,
    generation_config=config_traducao,
    return_full_text=False
)

llm_traducao = HuggingFacePipeline(
    pipeline=pipeline_traducao
)

print("Pipeline de tradução configurada.")
print(pipeline_traducao.generation_config)

In [ ]:
prompt_teste_traducao = """Traduza a pergunta médica abaixo do português para o inglês.

Retorne somente a tradução, sem explicações.

Pergunta:
Quais são os sintomas do câncer de mama?

Tradução:"""

teste_traducao = pipeline_traducao(
    prompt_teste_traducao,
    return_full_text=False
)

print(teste_traducao)

In [ ]:
# Corrige a configuração-base usada pelas pipelines

modelo.generation_config.max_length = None
modelo.generation_config.max_new_tokens = None

print(modelo.generation_config)

In [ ]:
import ipywidgets as widgets
from IPython.display import display


# ============================================================
# FUNÇÃO DE TRADUÇÃO
# ============================================================

def traduzir_pergunta(pergunta):

    prompt_traducao = f"""Traduza a pergunta médica abaixo do português para o inglês.

Regras:
- Retorne SOMENTE a tradução.
- Não explique a tradução.
- Preserve o significado médico da pergunta.

Pergunta:
{pergunta}

Tradução:"""

    mensagens_traducao = [
        {
            "role": "user",
            "content": prompt_traducao
        }
    ]

    prompt_chat = tokenizer.apply_chat_template(
        mensagens_traducao,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        prompt_chat,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(modelo.device)

    with torch.no_grad():
        saida = modelo.generate(
            **entradas,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    traducao = tokenizer.decode(
        saida[0][entradas["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return traducao


# ============================================================
# ASSISTENTE MÉDICO
# ============================================================

def executar_assistente(b):

    with saida:
        saida.clear_output()

        pergunta = caixa_pergunta.value.strip()

        if not pergunta:
            print("Digite uma pergunta.")
            return

        # ----------------------------------------------------
        # SYSTEM PROMPT
        # ----------------------------------------------------

        system_prompt = """Você é um assistente especializado em informação médica.

ESCOPO:
- Responda somente perguntas relacionadas à área médica e à saúde.
- Se a pergunta não estiver relacionada à área médica ou à saúde,
  não tente respondê-la.
- Para perguntas fora da sua área de atuação, responda exatamente:

"Não disponho de informações para responder a perguntas fora da área médica."

FONTES:
Para perguntas médicas, utilize somente:
1. o conhecimento adquirido durante o fine-tuning;
2. as informações explicitamente presentes no contexto médico fornecido pelo RAG.

REGRAS:
- Não invente informações.
- Não faça inferências médicas que não sejam sustentadas pelas fontes.
- Não utilize conhecimento externo às fontes disponíveis.
- Não acrescente informações desnecessárias.
- Ignore informações do contexto que não sejam relevantes para a pergunta.
- Responda sempre em português do Brasil.
- Seja objetivo e claro.

INSUFICIÊNCIA:
Se a pergunta for médica, mas as fontes disponíveis não fornecerem informação suficiente
para respondê-la, responda exatamente:

"Não há informações suficientes nas fontes disponíveis para responder a esta pergunta."
"""

        # ----------------------------------------------------
        # 1. TRADUÇÃO DA PERGUNTA
        # ----------------------------------------------------

        pergunta_busca = traduzir_pergunta(pergunta)

        # ----------------------------------------------------
        # 2. BUSCA NO RAG
        # ----------------------------------------------------

        documentos = buscar_medquad(
            pergunta_busca,
            k=5
        )

        if not documentos:
            print(
                "Não há informações suficientes nas fontes disponíveis "
                "para responder a esta pergunta."
            )
            return

        # ----------------------------------------------------
        # 3. CONTEXTO PRINCIPAL
        # ----------------------------------------------------

        documento_principal = documentos[0]
        contexto = documento_principal.page_content

        # ----------------------------------------------------
        # 4. CONVERSA NO FORMATO NATIVO DO QWEN
        # ----------------------------------------------------

        mensagens = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": f"""Contexto médico:
{contexto}

Pergunta do usuário:
{pergunta}"""
            }
        ]

        prompt_chat = tokenizer.apply_chat_template(
            mensagens,
            tokenize=False,
            add_generation_prompt=True
        )

        # ----------------------------------------------------
        # 5. TOKENIZAÇÃO
        # ----------------------------------------------------

        entradas = tokenizer(
            prompt_chat,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(modelo.device)

        # ----------------------------------------------------
        # 6. GERAÇÃO DA RESPOSTA
        # ----------------------------------------------------

        with torch.no_grad():
            saida_modelo = modelo.generate(
                **entradas,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        # ----------------------------------------------------
        # 7. EXTRAI SOMENTE A RESPOSTA
        # ----------------------------------------------------

        resposta = tokenizer.decode(
            saida_modelo[0][entradas["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        # ----------------------------------------------------
        # 8. FONTE
        # ----------------------------------------------------

        qid = documento_principal.metadata.get("qid", "N/A")
        focus = documento_principal.metadata.get("focus", "N/A")

        print("PERGUNTA:")
        print(pergunta)

        print("\nTRADUÇÃO PARA BUSCA:")
        print(pergunta_busca)

        print("\nRESPOSTA:")
        print(resposta)

        print("\nFONTE RAG:")
        print(f"- MedQuAD — {focus} (qid: {qid})")


# ============================================================
# INTERFACE
# ============================================================

caixa_pergunta = widgets.Textarea(
    placeholder="Digite sua pergunta aqui...",
    description="Pergunta:",
    layout=widgets.Layout(
        width="100%",
        height="100px"
    )
)

botao = widgets.Button(
    description="Perguntar"
)

saida = widgets.Output()

display(caixa_pergunta)
display(botao)
display(saida)

botao.on_click(executar_assistente)